# Step 2–3: Screening Eligibility, Test Assignment & Results

This notebook covers the screening layer of the simulation:
- **Step 2**: Determine which cancer screenings a patient is eligible for,
  assign the appropriate test modality, and handle unscreened patients.
- **Step 3**: Draw screening results (cervical: granular 5-category multinomial;
  lung: Lung-RADS 6-category draw).

All logic lives in `screening.py`. This notebook demonstrates and tests it.

**Handoff**: patients arrive here after being *seen* by a provider in
`01_arrivals.ipynb` (Sophia's layer). Results feed into `03_results_followup.ipynb`.

In [ ]:
import sys, random
sys.path.insert(0, '../src')   # ensure local .py files are importable

import config as cfg
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from patient import Patient
from population import sample_patient
from metrics import initialize_metrics, record_screening
from screening import (
    get_eligible_screenings,
    days_until_eligible,
    is_due_for_screening,
    get_cervical_age_stratum,
    assign_screening_test,
    draw_cervical_result,
    draw_lung_rads_result,
    run_screening_step,
)

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Eligibility Checks
Each cancer type has an eligibility function in `screening.py`.
`get_eligible_screenings(patient)` returns the full list for one patient.

In [ ]:
# Edge-case patients to verify eligibility logic
cases = [
    # (label, age, has_cervix, smoker, pack_years, bmi)
    ('Age 25, with cervix, non-smoker', 25, True,  False, 0,  25),
    ('Age 45, with cervix, smoker 25pk', 45, True,  True,  25, 25),
    ('Age 45, no cervix',               45, False, False, 0,  25),
    ('Age 55, smoker 25pk',             55, True,  True,  25, 25),
    ('Age 68, BMI 17',                  68, True,  False, 0,  17),
    ('Age 70, all eligible',            70, True,  True,  25, 25),
    ('Age 20, below all thresholds',    20, True,  False, 0,  25),
]

print(f'{"Label":<35}  {"Eligible for"}' )
print('-' * 70)
for label, age, has_cervix, smoker, pack_years, bmi in cases:
    p = sample_patient(0, 0, 'pcp', 'outpatient')
    p.age, p.has_cervix, p.smoker = age, has_cervix, smoker
    p.pack_years, p.bmi = pack_years, bmi
    eligible = get_eligible_screenings(p)
    print(f'{label:<35}  {", ".join(eligible) if eligible else "(none)"}')

## Test Assignment
Cervical test is age-stratified (ASCCP guidelines). Lung uses LDCT only.

In [ ]:
ages = [22, 35, 50, 66]
print(f'{"Age":<6}  {"Stratum":<10}  {"Assigned test"}')
print('-' * 40)
for age in ages:
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix = age, True
    stratum = get_cervical_age_stratum(age)
    test    = assign_screening_test(p, 'cervical')
    print(f'{age:<6}  {stratum:<10}  {test}')

## Cervical Result Distribution
Draw 5,000 results for each test type and verify the distribution
matches the config probabilities.

In [ ]:
N = 5_000
p_template = sample_patient(0, 0, 'gynecologist', 'outpatient')
p_template.age, p_template.has_cervix = 42, True
p_template.hpv_positive, p_template.prior_cin = False, None

# ── Cytology (age 30-65) ──────────────────────────────────────────────────────
results_cyto = [draw_cervical_result(p_template, 'cytology') for _ in range(N)]
counts_cyto  = Counter(results_cyto)

print(f'Cytology result distribution (n={N:,}, age=42):')
for cat in ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL']:
    cnt      = counts_cyto.get(cat, 0)
    expected = cfg.CERVICAL_RESULT_PROBS['middle_cytology'].get(cat, 0)
    print(f'  {cat:<30} observed={cnt/N*100:5.1f}%  expected={expected*100:5.1f}%')

print()

# ── HPV-alone (age 30-65) ─────────────────────────────────────────────────────
results_hpv = [draw_cervical_result(p_template, 'hpv_alone') for _ in range(N)]
counts_hpv  = Counter(results_hpv)

print(f'HPV-alone result distribution (n={N:,}, age=42):')
for cat in ['HPV_NEGATIVE', 'HPV_POSITIVE']:
    cnt      = counts_hpv.get(cat, 0)
    expected = cfg.CERVICAL_RESULT_PROBS['middle_hpv'].get(cat, 0)
    print(f'  {cat:<30} observed={cnt/N*100:5.1f}%  expected={expected*100:5.1f}%')

## Cervical Result Distributions by Age Stratum
Compare cytology abnormality rates across young (21–29) and middle-aged (30–65) strata.

In [ ]:
N = 10_000
strata_info = [
    ('young (age 25)',  25, 'cytology',  'young'),
    ('middle (age 42)', 42, 'cytology', 'middle_cytology'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
categories = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL']

for ax, (label, age, test, cfg_key) in zip(axes, strata_info):
    pt = sample_patient(0, 0, 'gynecologist', 'outpatient')
    pt.age, pt.has_cervix, pt.hpv_positive, pt.prior_cin = age, True, False, None
    results = [draw_cervical_result(pt, test) for _ in range(N)]
    counts  = Counter(results)
    obs  = [counts.get(c, 0) / N * 100 for c in categories]
    exp  = [cfg.CERVICAL_RESULT_PROBS[cfg_key].get(c, 0) * 100 for c in categories]
    x = range(len(categories))
    ax.bar([i - 0.2 for i in x], obs, width=0.4, label='Observed', alpha=0.8)
    ax.bar([i + 0.2 for i in x], exp, width=0.4, label='Expected', alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(categories, rotation=30)
    ax.set_ylabel('Frequency (%)')
    ax.set_title(f'Cytology — {label}')
    ax.legend()

plt.tight_layout()
plt.show()

## Risk Factor Effect on Cytology Results
HPV-positive and prior high-grade CIN inflate abnormal result categories.

In [ ]:
N = 10_000
abnormal_cats = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL']

groups = [
    ('No risk factors',    False, None),
    ('HPV positive',       True,  None),
    ('Prior CIN2/3',       False, 'CIN2'),
    ('HPV+ & Prior CIN3',  True,  'CIN3'),
]

labels, abnormal_rates = [], []
for label, hpv_pos, prior_cin in groups:
    pt = sample_patient(0, 0, 'gynecologist', 'outpatient')
    pt.age, pt.has_cervix = 42, True
    pt.hpv_positive, pt.prior_cin = hpv_pos, prior_cin
    results = [draw_cervical_result(pt, 'cytology') for _ in range(N)]
    rate = sum(1 for r in results if r in abnormal_cats) / N * 100
    labels.append(label)
    abnormal_rates.append(rate)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, abnormal_rates, color=['steelblue', 'tomato', 'orange', 'firebrick'])
ax.set_ylabel('Any abnormal result (%)')
ax.set_title('Risk factor effect on cervical cytology abnormality rate')
ax.set_ylim(0, max(abnormal_rates) * 1.3)
for i, v in enumerate(abnormal_rates):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Lung-RADS Distribution
Verify the Lung-RADS draw matches `config.LUNG_RADS_PROBS`.

In [ ]:
N = 10_000
rads_results = [draw_lung_rads_result() for _ in range(N)]
rads_counts  = Counter(rads_results)
categories   = list(cfg.LUNG_RADS_PROBS.keys())

obs = [rads_counts.get(c, 0) / N * 100 for c in categories]
exp = [cfg.LUNG_RADS_PROBS[c] * 100 for c in categories]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(categories))
ax.bar([i - 0.2 for i in x], obs, width=0.4, label='Observed', alpha=0.8)
ax.bar([i + 0.2 for i in x], exp, width=0.4, label='Expected', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(categories)
ax.set_ylabel('Frequency (%)')
ax.set_title(f'Lung-RADS distribution (n={N:,})')
ax.legend()
plt.tight_layout()
plt.show()

## Smoke Test — Run 30 Patients Through Screening
Simulates one provider-seen day: generate patients, run `run_screening_step`
for each eligible cancer, collect results.

In [ ]:
random.seed(42)
DAY        = 0
N_PATIENTS = 30
metrics    = initialize_metrics()

patients      = [sample_patient(i, DAY, 'gynecologist', 'outpatient') for i in range(N_PATIENTS)]
screening_log = defaultdict(list)

for p in patients:
    metrics['n_patients'] += 1
    eligible = get_eligible_screenings(p)

    if not eligible:
        # Three-way eligibility split (mirrors runner._screen_patient logic)
        soonest = -1
        for cancer in cfg.ACTIVE_CANCERS:
            d = days_until_eligible(p, cancer)
            if d > 0 and (soonest < 0 or d < soonest):
                soonest = d
        if soonest > 0:
            screening_log['not_yet_eligible'].append(f'return in {soonest}d')
        else:
            screening_log['permanently_ineligible'].append('exit')
        metrics['n_unscreened'] += 1
        continue

    metrics['n_eligible_any'] += 1
    for cancer in eligible:
        result = run_screening_step(p, cancer, DAY, metrics)
        if result:
            # record_screening is normally called by the runner; we call it
            # explicitly here because we are operating outside the runner.
            record_screening(metrics, p, cancer, result)
            screening_log[cancer].append(result)

print(f'Results across {N_PATIENTS} patients:\n')
for key, results in sorted(screening_log.items()):
    cnt = Counter(results)
    print(f'  {key}:')
    for r, c in sorted(cnt.items()):
        print(f'    {r:<35} {c}')

print(f'\n  Total seen:                       {metrics["n_patients"]}')
print(f'  Eligible for any screen:          {metrics["n_eligible_any"]}')
print(f'  Not yet / permanently ineligible: {metrics["n_unscreened"]}')
print(f'  Cervical screened:                {metrics["n_screened"]["cervical"]}')
print(f'  Lung screened:                    {metrics["n_screened"]["lung"]}')

print('\n── Sample patient trace ──')
patients[0].print_history()